In [28]:
%cd /content/modCudaFunctions
!rm -rf build
!mkdir build
%cd build
!cmake ..
!make

/content/modCudaFunctions
/content/modCudaFunctions/build
-- The CXX compiler identification is GNU 11.4.0
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
-- Configuring done (2.5s)
CMake Warning (dev) in CMakeLists.txt:
  Policy CMP0104 is not set: CMAKE_CUDA_ARCHITECTURES now detected for NVCC,
  empty CUDA_ARCHITECTURES not allowed.  Run "cmake --help-policy CMP0104"
  for policy details.  Use the cmake_policy command to set the policy and
  suppress this warning.

  CUDA_ARCHITECTURES is empty for target "vector_add".

In [29]:
%cd /content/modCudaFunctions/python/wrapper
!rm -rf build *.so
!python3 setup.py build_ext --inplace

/content/modCudaFunctions/python/wrapper
running build_ext
building 'vector_add_wrapper' extension
creating build/temp.linux-x86_64-cpython-312
x86_64-linux-gnu-gcc -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -fPIC -I/usr/include/python3.12 -c vector_add_wrapper.c -o build/temp.linux-x86_64-cpython-312/vector_add_wrapper.o
creating build/lib.linux-x86_64-cpython-312
x86_64-linux-gnu-gcc -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 build/temp.linux-x86_64-cpython-312/vector_add_wrapper.o -L/usr/lib/x86_64-linux-gnu -ldl -o build/lib.linux-x86_64-cpython-312/vector_add_wrapper.cpython-312-x86_64-linux-gnu.so
copying build/lib.linux-x86_64-cpython-312/vector_add_wrapper.cpython-312-x86_64-linux-gnu.so -> 


In [30]:
%cd /content/modCudaFunctions/python/wrapper_np
!rm -rf build *.so
!python3 setup.py build_ext --inplace

/content/modCudaFunctions/python/wrapper_np
running build_ext
building 'vector_add_np_wrapper' extension
creating build/temp.linux-x86_64-cpython-312
x86_64-linux-gnu-gcc -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -fPIC -I/usr/local/lib/python3.12/dist-packages/numpy/_core/include -I/usr/include/python3.12 -c vector_add_np_wrapper.c -o build/temp.linux-x86_64-cpython-312/vector_add_np_wrapper.o
creating build/lib.linux-x86_64-cpython-312
x86_64-linux-gnu-gcc -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 build/temp.linux-x86_64-cpython-312/vector_add_np_wrapper.o -L/usr/lib/x86_64-linux-gnu -ldl -o build/lib.linux-x86_64-cpython-312/vector_add_np_wrapper.cpython-312-x86_64-linux-gnu.so
copying build/lib.linux-x86_64-cpython-312/vector_add_np_wrapper.cpython-312-x86_64-linux-gnu.so -> 


In [49]:
%%writefile /content/modCudaFunctions/python/ctypes/test_ctypes.py
import ctypes
import numpy as np

lib = ctypes.CDLL("/content/modCudaFunctions/build/libvector_add.so")

lib.vectorAdd.argtypes = [
    ctypes.POINTER(ctypes.c_int),
    ctypes.POINTER(ctypes.c_int),
    ctypes.POINTER(ctypes.c_int),
    ctypes.c_int
]

lib.vectorAdd.restype = None

N = 10

a = np.arange(N, dtype=np.int32)
b = np.arange(N, 2 * N, dtype=np.int32)
c = np.zeros(N, dtype=np.int32)

lib.vectorAdd(
    a.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
    b.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
    c.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
    N
)

print("Input A:", a)
print("Input B:", b)
print("Output :", c)

Overwriting /content/modCudaFunctions/python/ctypes/test_ctypes.py


In [50]:
%cd /content/modCudaFunctions/python/ctypes
!python3 test.py 10

/content/modCudaFunctions/python/ctypes
Method 1: ctypes test.py
Input A: [0 1 2 3 4 5 6 7 8 9]
Input B: [10 11 12 13 14 15 16 17 18 19]
Result : [10 12 14 16 18 20 22 24 26 28]


In [ ]:
%%writefile /content/modCudaFunctions/python/ctypes/test.py
import sys
import ctypes
import numpy as np

if len(sys.argv) != 2:
    print("Usage: python3 test.py <N>")
    sys.exit(1)

N = int(sys.argv[1])

lib = ctypes.CDLL("/content/modCudaFunctions/build/libvector_add.so")

lib.vectorAdd.argtypes = [
    ctypes.POINTER(ctypes.c_int),
    ctypes.POINTER(ctypes.c_int),
    ctypes.POINTER(ctypes.c_int),
    ctypes.c_int
]

lib.vectorAdd.restype = None

a = np.arange(N, dtype=np.int32)
b = np.arange(N, 2 * N, dtype=np.int32)
c = np.zeros(N, dtype=np.int32)

lib.vectorAdd(
    a.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
    b.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
    c.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
    N
)

print("Input A:", a)
print("Input B:", b)
print("Output :", c)

Writing /content/modCudaFunctions/python/ctypes/test.py


In [ ]:
!python3 test_ctypes.py

Method 1: ctypes
Input A: [0 1 2 3 4 5 6 7 8 9]
Input B: [10 11 12 13 14 15 16 17 18 19]
Result : [10 12 14 16 18 20 22 24 26 28]


In [35]:
%%writefile /content/modCudaFunctions/python/wrapper/vector_add_wrapper.c
#define PY_SSIZE_T_CLEAN
#include <Python.h>
#include <dlfcn.h>
#include <stdlib.h>

typedef void (*vectorAddFunc)(int *, int *, int *, int);

static vectorAddFunc vectorAdd = NULL;

static PyObject* pyVectorAdd(PyObject* self, PyObject* args) {
    PyObject *py_a, *py_b;
    int N;

    if (!PyArg_ParseTuple(args, "OOi", &py_a, &py_b, &N))
        return NULL;

    if (!PyList_Check(py_a) || !PyList_Check(py_b) ||
        PyList_Size(py_a) != N || PyList_Size(py_b) != N) {
        PyErr_SetString(PyExc_ValueError, "Arguments must be lists of size N.");
        return NULL;
    }

    int *a = malloc(N * sizeof(int));
    int *b = malloc(N * sizeof(int));
    int *c = malloc(N * sizeof(int));

    for (int i = 0; i < N; i++) {
        a[i] = (int)PyLong_AsLong(PyList_GetItem(py_a, i));
        b[i] = (int)PyLong_AsLong(PyList_GetItem(py_b, i));
    }

    vectorAdd(a, b, c, N);

    PyObject *result = PyList_New(N);

    for (int i = 0; i < N; i++) {
        PyList_SetItem(result, i, PyLong_FromLong(c[i]));
    }

    free(a);
    free(b);
    free(c);

    return result;
}

static PyMethodDef VectorAddMethods[] = {
    {"vectorAdd", pyVectorAdd, METH_VARARGS, "Run CUDA vector addition"},
    {NULL, NULL, 0, NULL}
};

static struct PyModuleDef vectoraddmodule = {
    PyModuleDef_HEAD_INIT,
    "vector_add_wrapper",
    NULL,
    -1,
    VectorAddMethods
};

PyMODINIT_FUNC PyInit_vector_add_wrapper(void) {
    void *handle = dlopen(
        "/content/modCudaFunctions/build/libvector_add.so",
        RTLD_LAZY
    );

    if (!handle) {
        PyErr_SetString(PyExc_ImportError, dlerror());
        return NULL;
    }

    vectorAdd = (vectorAddFunc)dlsym(handle, "vectorAdd");

    if (!vectorAdd) {
        PyErr_SetString(PyExc_ImportError, dlerror());
        return NULL;
    }

    return PyModule_Create(&vectoraddmodule);
}

Overwriting /content/modCudaFunctions/python/wrapper/vector_add_wrapper.c


In [36]:
%%writefile /content/modCudaFunctions/python/wrapper/setup.py
from setuptools import setup, Extension

module = Extension(
    "vector_add_wrapper",
    sources=["vector_add_wrapper.c"],
    libraries=["dl"]
)

setup(
    name="vector_add_wrapper",
    version="1.0",
    ext_modules=[module]
)

Overwriting /content/modCudaFunctions/python/wrapper/setup.py


In [37]:
%%writefile /content/modCudaFunctions/python/wrapper/test_vector_add_wrapper.py
import vector_add_wrapper as vaw

N = 10

a = list(range(N))
b = list(range(N, 2 * N))

result = vaw.vectorAdd(a, b, N)

print("Input A:", a)
print("Input B:", b)
print("Output :", result)

Overwriting /content/modCudaFunctions/python/wrapper/test_vector_add_wrapper.py


In [38]:
%%writefile /content/modCudaFunctions/python/wrapper/test.py
import sys
import vector_add_wrapper as vaw

if len(sys.argv) != 2:
    print("Usage: python3 test.py <N>")
    sys.exit(1)

N = int(sys.argv[1])

a = list(range(N))
b = list(range(N, 2 * N))

result = vaw.vectorAdd(a, b, N)

print("Input A:", a)
print("Input B:", b)
print("Output :", result)

Overwriting /content/modCudaFunctions/python/wrapper/test.py


In [39]:
%cd /content/modCudaFunctions/python/wrapper
!rm -rf build
!rm -f vector_add_wrapper*.so
!python3 setup.py build_ext --inplace

/content/modCudaFunctions/python/wrapper
running build_ext
building 'vector_add_wrapper' extension
creating build/temp.linux-x86_64-cpython-312
x86_64-linux-gnu-gcc -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -fPIC -I/usr/include/python3.12 -c vector_add_wrapper.c -o build/temp.linux-x86_64-cpython-312/vector_add_wrapper.o
creating build/lib.linux-x86_64-cpython-312
x86_64-linux-gnu-gcc -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 build/temp.linux-x86_64-cpython-312/vector_add_wrapper.o -L/usr/lib/x86_64-linux-gnu -ldl -o build/lib.linux-x86_64-cpython-312/vector_add_wrapper.cpython-312-x86_64-linux-gnu.so
copying build/lib.linux-x86_64-cpython-312/vector_add_wrapper.cpython-312-x86_64-linux-gnu.so -> 


In [40]:
!python3 test.py 10

Input A: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Input B: [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Output : [10, 12, 14, 16, 18, 20, 22, 24, 26, 28]


In [41]:
!python3 test_vector_add_wrapper.py

Input A: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Input B: [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Output : [10, 12, 14, 16, 18, 20, 22, 24, 26, 28]


In [42]:
%%writefile /content/modCudaFunctions/python/wrapper_np/vector_add_np_wrapper.c
#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION
#define PY_SSIZE_T_CLEAN

#include <Python.h>
#include <numpy/arrayobject.h>
#include <dlfcn.h>

typedef void (*vectorAddFunc)(int *, int *, int *, int);

static vectorAddFunc vectorAdd = NULL;

static PyObject* pyVectorAdd(PyObject* self, PyObject* args) {
    PyArrayObject *a;
    PyArrayObject *b;
    PyArrayObject *c;
    int N;

    if (!PyArg_ParseTuple(
        args,
        "O!O!O!i",
        &PyArray_Type, &a,
        &PyArray_Type, &b,
        &PyArray_Type, &c,
        &N
    )) {
        return NULL;
    }

    if (PyArray_TYPE(a) != NPY_INT32 ||
        PyArray_TYPE(b) != NPY_INT32 ||
        PyArray_TYPE(c) != NPY_INT32) {
        PyErr_SetString(PyExc_TypeError, "Arrays must use dtype int32.");
        return NULL;
    }

    if (PyArray_SIZE(a) != N ||
        PyArray_SIZE(b) != N ||
        PyArray_SIZE(c) != N) {
        PyErr_SetString(PyExc_ValueError, "Arrays must have size N.");
        return NULL;
    }

    int *a_ptr = (int*)PyArray_DATA(a);
    int *b_ptr = (int*)PyArray_DATA(b);
    int *c_ptr = (int*)PyArray_DATA(c);

    vectorAdd(a_ptr, b_ptr, c_ptr, N);

    Py_RETURN_NONE;
}

static PyMethodDef VectorAddMethods[] = {
    {"vectorAdd", pyVectorAdd, METH_VARARGS, "Run CUDA vector addition with NumPy"},
    {NULL, NULL, 0, NULL}
};

static struct PyModuleDef vectoraddmodule = {
    PyModuleDef_HEAD_INIT,
    "vector_add_np_wrapper",
    NULL,
    -1,
    VectorAddMethods
};

PyMODINIT_FUNC PyInit_vector_add_np_wrapper(void) {
    void *handle = dlopen(
        "/content/modCudaFunctions/build/libvector_add.so",
        RTLD_LAZY
    );

    if (!handle) {
        PyErr_SetString(PyExc_ImportError, dlerror());
        return NULL;
    }

    vectorAdd = (vectorAddFunc)dlsym(handle, "vectorAdd");

    if (!vectorAdd) {
        PyErr_SetString(PyExc_ImportError, dlerror());
        return NULL;
    }

    import_array();

    return PyModule_Create(&vectoraddmodule);
}

Overwriting /content/modCudaFunctions/python/wrapper_np/vector_add_np_wrapper.c


In [43]:
%%writefile /content/modCudaFunctions/python/wrapper_np/setup.py
from setuptools import setup, Extension
import numpy

module = Extension(
    "vector_add_np_wrapper",
    sources=["vector_add_np_wrapper.c"],
    include_dirs=[numpy.get_include()],
    libraries=["dl"]
)

setup(
    name="vector_add_np_wrapper",
    version="1.0",
    ext_modules=[module]
)

Overwriting /content/modCudaFunctions/python/wrapper_np/setup.py


In [44]:
%%writefile /content/modCudaFunctions/python/wrapper_np/test_vector_add_np_wrapper.py
import numpy as np
import vector_add_np_wrapper as vaw_np

N = 10

a = np.arange(N, dtype=np.int32)
b = np.arange(N, 2 * N, dtype=np.int32)
c = np.zeros(N, dtype=np.int32)

vaw_np.vectorAdd(a, b, c, N)

print("Input A:", a)
print("Input B:", b)
print("Output :", c)

Overwriting /content/modCudaFunctions/python/wrapper_np/test_vector_add_np_wrapper.py


In [45]:
%%writefile /content/modCudaFunctions/python/wrapper_np/test.py
import sys
import numpy as np
import vector_add_np_wrapper as vaw_np

if len(sys.argv) != 2:
    print("Usage: python3 test.py <N>")
    sys.exit(1)

N = int(sys.argv[1])

a = np.arange(N, dtype=np.int32)
b = np.arange(N, 2 * N, dtype=np.int32)
c = np.zeros(N, dtype=np.int32)

vaw_np.vectorAdd(a, b, c, N)

print("Input A:", a)
print("Input B:", b)
print("Output :", c)

Overwriting /content/modCudaFunctions/python/wrapper_np/test.py


In [46]:
%cd /content/modCudaFunctions/python/wrapper_np
!rm -rf build
!rm -f vector_add_np_wrapper*.so
!python3 setup.py build_ext --inplace

/content/modCudaFunctions/python/wrapper_np
running build_ext
building 'vector_add_np_wrapper' extension
creating build/temp.linux-x86_64-cpython-312
x86_64-linux-gnu-gcc -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -fPIC -I/usr/local/lib/python3.12/dist-packages/numpy/_core/include -I/usr/include/python3.12 -c vector_add_np_wrapper.c -o build/temp.linux-x86_64-cpython-312/vector_add_np_wrapper.o
creating build/lib.linux-x86_64-cpython-312
x86_64-linux-gnu-gcc -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 build/temp.linux-x86_64-cpython-312/vector_add_np_wrapper.o -L/usr/lib/x86_64-linux-gnu -ldl -o build/lib.linux-x86_64-cpython-312/vector_add_np_wrapper.cpython-312-x86_64-linux-gnu.so
copying build/lib.linux-x86_64-cpython-312/vector_add_np_wrapper.cpython-312-x86_64-linux-gnu.so -> 


In [47]:
!python3 test.py 10

Input A: [0 1 2 3 4 5 6 7 8 9]
Input B: [10 11 12 13 14 15 16 17 18 19]
Output : [10 12 14 16 18 20 22 24 26 28]


In [48]:
!python3 test_vector_add_np_wrapper.py

Input A: [0 1 2 3 4 5 6 7 8 9]
Input B: [10 11 12 13 14 15 16 17 18 19]
Output : [10 12 14 16 18 20 22 24 26 28]
